In [5]:
import glob 
import json

dataset_dir = '/home/jovyan/datasets/mpt_train_llama_10k_cc'
p = glob.glob(f'{dataset_dir}/*/')

allowed = '''FRW_llama_ext_10k
arxiv
book
librusec_fb
ru_cc_ver2_llama_ext_10k
stackexchange
starcoder
wikipedia
libgen_llama_ext_10k
wikiquote_llama_ext_10k'''.split()

size = {}
for path in p:
    shards = json.load(open(path + 'index.json'))['shards']
    # print(shards[0])
    name = path[len(f'{dataset_dir}/'):-1]
    if name in allowed:
        size[name] = len(shards) * shards[0]['samples'] * 4096 / 1e9
    
size, len(size), len(allowed)

({'book': 29.488328704,
  'arxiv': 30.637006848,
  'librusec_fb': 43.197005824,
  'stackexchange': 22.369878016,
  'wikipedia': 24.06354944,
  'starcoder': 113.31667968,
  'wikiquote_llama_ext_10k': 0.050307072,
  'ru_cc_ver2_llama_ext_10k': 200.649756672,
  'FRW_llama_ext_10k': 691.814469632,
  'libgen_llama_ext_10k': 87.274385408},
 10,
 10)

In [16]:
from transformers import AutoTokenizer

In [17]:
tokenizer_path = '/home/jovyan/models/llama2-7b-fp16-TheBloke-extended_tok-cc_10k'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, legacy=True, use_fast=False)

In [18]:
tokenizer

LlamaTokenizer(name_or_path='/home/jovyan/models/llama2-7b-fp16-TheBloke-extended_tok-cc_10k', vocab_size=42000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True), 'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True), 'unk_token': AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True)}, clean_up_tokenization_spaces=False)

In [79]:
from streaming import LocalDataset
from torch.utils.data import DataLoader
from streaming import StreamingDataset

import numpy as np
import random

In [76]:
DATASETS = {}
for domain in allowed:
    DATASETS[domain] = LocalDataset(local=f'{dataset_dir}/{domain}')

In [85]:
for domain in allowed:
    dataset = DATASETS[domain]
    is_eos_ok = False
    i = 100
    while not is_eos_ok or i > 0:
        item = dataset.get_item(random.randint(0, len(dataset)))
        deserialized_bytes = np.frombuffer(item['tokens'], dtype=np.uint64)
        is_eos_ok = (2 in set(deserialized_bytes))
        i -= 1
        
    item = dataset.get_item(0)
    deserialized_bytes = np.frombuffer(item['tokens'], dtype=np.uint64)
    # deserialized_bytes
    decoded = tokenizer.decode(deserialized_bytes)
    print(f"{domain.upper()} context {len(deserialized_bytes)} eos={is_eos_ok}:\n\n{decoded[:20]}\n\n")
    print('---'*20)

FRW_LLAMA_EXT_10K context 4096 eos=True:

<s> The Silver Machi


------------------------------------------------------------
ARXIV context 4096 eos=True:

<s>\section{Introduc


------------------------------------------------------------
BOOK context 4096 eos=True:

<s>













The


------------------------------------------------------------
LIBRUSEC_FB context 4096 eos=True:

<s> Тисецкий Григори


------------------------------------------------------------
RU_CC_VER2_LLAMA_EXT_10K context 4096 eos=True:

<s> Циклонные сепара


------------------------------------------------------------
STACKEXCHANGE context 4096 eos=True:

<s> Q: What is the d


------------------------------------------------------------
STARCODER context 4096 eos=True:

<s> <reponame>adamve


------------------------------------------------------------
WIKIPEDIA context 4096 eos=True:

<s> Григорианският к


------------------------------------------------------------
LIBGEN_LLAMA_EXT_10K context 4096 eo